# Pricing fundamentals: instrument JSON, market context, and `ValuationResult`

**Prerequisites:** work through the foundation notebooks **01** (core types and money), **02** (dates, calendars, schedules), and **03** (market data and curves) in this curriculum. This notebook assumes you are comfortable with `DiscountCurve`, `MarketContext`, and basic `finstack` imports.

## Concepts: the instrument pricing pipeline

End-to-end pricing in finstack follows a small, explicit pipeline:

1. **Instrument** — Describe the trade as **tagged JSON** (`type` + `spec`). Validation normalizes the payload to a canonical form the Rust loader accepts.
2. **Market** — Provide curves (and optionally FX) inside a **`MarketContext`**, then **serialize it to JSON** so the same snapshot can cross process boundaries or be logged.
3. **Valuation date** — An ISO calendar date (`YYYY-MM-DD`) marking *as of*.
4. **Model** — A string **model key** (for example `discounting` for simple PV off a discount curve).
5. **Result** — The pricer returns **`ValuationResult` as JSON**; parse it with `ValuationResult.from_json` to read PV, currency, and any requested **metrics**.

Optional **metrics** are requested by name. The registry knows which measures apply to which instrument; names you request that are not implemented for that instrument are simply omitted from the output.

### Imports

Core entry points live in `finstack.valuations` alongside `ValuationResult`.

In [10]:
import json

from finstack.valuations import (
    ValuationResult,
    list_standard_metrics,
    list_standard_metrics_grouped,
    price_instrument,
    price_instrument_with_metrics,
    validate_instrument_json,
)

print("Imported pricing helpers and ValuationResult.")

Imported pricing helpers and ValuationResult.


### Discover metrics: `list_standard_metrics_grouped`

Metrics are organized into groups so you can find what you need quickly. Use `list_standard_metrics_grouped()` to see all metrics by category, or `list_standard_metrics()` for a flat alphabetical list.

In [11]:
grouped = list_standard_metrics_grouped()
total = sum(len(v) for v in grouped.values())
print(f"Standard metrics ({total}) across {len(grouped)} groups:\n")
for group, metrics in grouped.items():
    print(f"  {group} ({len(metrics)})")
    for m in metrics:
        print(f"    {m}")

Standard metrics (150) across 10 groups:

  Rates (18)
    annuity
    annuity_primary
    annuity_reference
    basis_par_spread
    deposit_par_rate
    df_end
    df_end_from_quote
    df_start
    financing_annuity
    incremental_par_spread
    index_delta
    par_rate
    pv_fixed
    pv_float
    pv_primary
    pv_reference
    quote_rate
    yf
  Sensitivity (16)
    bucketed_dv01
    collateral_haircut01
    collateral_price01
    conversion01
    convexity_adjustment_risk
    dividend01
    duration_dv01
    dv01
    dv01_domestic
    dv01_foreign
    foreign_rho
    forward_pv01
    inflation01
    npv01
    pv01
    rho
  Greeks (19)
    bucketed_vega
    charm
    color
    cross_gamma_fx_rates
    cross_gamma_fx_vol
    cross_gamma_rates_credit
    cross_gamma_spot_credit
    cross_gamma_spot_vol
    cs_gamma
    delta
    gamma
    inflation_convexity
    ir_convexity
    ir_cross_gamma
    speed
    vanna
    variance_vega
    vega
    volga
  FX (6)
    base_amount
   

### Validate instrument JSON: `validate_instrument_json`

Pass the raw JSON **string**. The function returns canonical, pretty-printed JSON suitable for `price_instrument`.

In [12]:
instrument = json.dumps(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-1",
            "notional": {"amount": 1000000.0, "currency": "USD"},
            "start_date": "2025-01-15",
            "maturity": "2025-06-15",
            "day_count": "Act360",
            "quote_rate": 0.05,
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)
canonical = validate_instrument_json(instrument)
print(canonical)

{
  "type": "deposit",
  "spec": {
    "id": "DEP-1",
    "notional": {
      "amount": "1000000",
      "currency": "USD"
    },
    "start_date": "2025-01-15",
    "maturity": "2025-06-15",
    "day_count": "Act360",
    "quote_rate": "0.05",
    "discount_curve_id": "USD-OIS",
    "pricing_overrides": {
      "quoted_clean_price": null,
      "implied_volatility": null,
      "quoted_spread_bp": null,
      "upfront_payment": null,
      "rho_bump_decimal": null,
      "vega_bump_decimal": null,
      "ytm_bump_decimal": null,
      "spot_bump_pct": null,
      "vol_bump_pct": null,
      "rate_bump_bp": null,
      "credit_spread_bump_bp": null,
      "adaptive_bumps": false,
      "mc_seed_scenario": null,
      "theta_period": null,
      "vol_surface_extrapolation": "error",
      "tree_steps": null,
      "tree_volatility": null,
      "use_gobet_miri": false,
      "call_friction_cents": null,
      "mean_reversion": null
    },
    "attributes": {},
    "spot_lag_days": null,

### Market context as JSON

Build a `MarketContext`, insert a `DiscountCurve` (id must match the instrument's `discount_curve_id`), then call **`to_json()`** to produce the string passed into pricing.

In [13]:
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext

mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75)],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
print(f"market_json length: {len(market_json)} chars")
print(market_json[:500] + ("..." if len(market_json) > 500 else ""))

market_json length: 756 chars
{
  "version": 2,
  "curves": [
    {
      "type": "discount",
      "id": "USD-OIS",
      "base": "2025-01-15",
      "day_count": "Act365F",
      "knot_points": [
        [
          0.0,
          1.0
        ],
        [
          0.5,
          0.975
        ],
        [
          1.0,
          0.95
        ],
        [
          5.0,
          0.75
        ]
      ],
      "interp_style": "monotone_convex",
      "extrapolation": "flat_forward",
      "min_forward_rate": null,
      "a...


### Price: `price_instrument`

Arguments: canonical instrument JSON, market JSON, **as_of** date string, and optional **`model`** (default `discounting`). The return value is a JSON string; pretty-printed.

**PV sign:** A **negative** present value means a **cash outflow** from the perspective of the **position holder** encoded in the instrument (e.g. paying fixed / lending cash in a deposit), consistent with the signed `value.amount` in the JSON envelope.

In [14]:
result_json = price_instrument(canonical, market_json, "2025-01-15", model="discounting")
print(result_json)

{
  "instrument_id": "DEP-1",
  "as_of": "2025-01-15",
  "value": {
    "amount": "-145.5801270729832440481613",
    "currency": "USD"
  },
  "measures": {},
  "meta": {
    "numeric_mode": "F64",
    "rounding": {
      "mode": "Bankers",
      "ingest_scale_by_ccy": {},
      "output_scale_by_ccy": {},
      "tolerances": {
        "rate_epsilon": 1e-12,
        "generic_epsilon": 1e-10
      },
      "version": 1
    },
    "fx_policy_applied": null,
    "timestamp": "+002026-04-13T01:00:45.580493000Z",
    "version": "0.4.1"
  },
  "covenants": null
}


### Price with metrics: `price_instrument_with_metrics`

Pass a list of metric names. **Only metrics registered for that instrument type appear** in `measures` — for example, a deposit may expose `dv01` but not `duration` or `convexity`.

In [15]:
result_json = price_instrument_with_metrics(
    canonical,
    market_json,
    "2025-01-15",
    model="discounting",
    metrics=["dv01", "duration", "convexity","bucketed_dv01"],
)
vr = ValuationResult.from_json(result_json)
print("Requested: dv01, duration, convexity")
print("Returned metric keys:", vr.metric_keys())

Requested: dv01, duration, convexity
Returned metric keys: ['dv01', 'bucketed_dv01', 'bucketed_dv01::USD-OIS::10y', 'bucketed_dv01::USD-OIS::15y', 'bucketed_dv01::USD-OIS::1y', 'bucketed_dv01::USD-OIS::20y', 'bucketed_dv01::USD-OIS::2y', 'bucketed_dv01::USD-OIS::30y', 'bucketed_dv01::USD-OIS::3m', 'bucketed_dv01::USD-OIS::3y', 'bucketed_dv01::USD-OIS::5y', 'bucketed_dv01::USD-OIS::6m', 'bucketed_dv01::USD-OIS::7y']


### Parse results: `ValuationResult`

Use **`from_json`** on the string returned by the pricers. The PV amount is the **`price`** property; **`get_metric(key)`** reads individual scalars.

**PV sign (again):** `ValuationResult.price` uses the same convention — **negative PV** is an **outflow** for the holder of the priced position (compare the deposit example above).

In [16]:
vr = ValuationResult.from_json(result_json)
print(f"Instrument: {vr.instrument_id}")
print(f"Price: {vr.price:.2f}")
print(f"Currency: {vr.currency}")
print(f"Metric keys: {vr.metric_keys()}")
print(f"Metric count: {vr.metric_count()}")
print(f"Covenants passed: {vr.all_covenants_passed()}")
for key in vr.metric_keys():
    print(f"  {key}: {vr.get_metric(key)}")

Instrument: DEP-1
Price: -145.58
Currency: USD
Metric keys: ['dv01', 'bucketed_dv01', 'bucketed_dv01::USD-OIS::10y', 'bucketed_dv01::USD-OIS::15y', 'bucketed_dv01::USD-OIS::1y', 'bucketed_dv01::USD-OIS::20y', 'bucketed_dv01::USD-OIS::2y', 'bucketed_dv01::USD-OIS::30y', 'bucketed_dv01::USD-OIS::3m', 'bucketed_dv01::USD-OIS::3y', 'bucketed_dv01::USD-OIS::5y', 'bucketed_dv01::USD-OIS::6m', 'bucketed_dv01::USD-OIS::7y']
Metric count: 13
Covenants passed: True
  dv01: -41.363840395655274
  bucketed_dv01: -41.36384040279638
  bucketed_dv01::USD-OIS::10y: 0.0
  bucketed_dv01::USD-OIS::15y: 0.0
  bucketed_dv01::USD-OIS::1y: 7.1395121758976785
  bucketed_dv01::USD-OIS::20y: 0.0
  bucketed_dv01::USD-OIS::2y: 0.0
  bucketed_dv01::USD-OIS::30y: 0.0
  bucketed_dv01::USD-OIS::3m: 0.0
  bucketed_dv01::USD-OIS::3y: 0.0
  bucketed_dv01::USD-OIS::5y: 0.0
  bucketed_dv01::USD-OIS::6m: -48.503352578694056
  bucketed_dv01::USD-OIS::7y: 0.0


### Model keys and metric naming

**Model keys** (string literals accepted by `price_instrument`):

- `discounting`, `black76`, `hazard_rate`, `hull_white_1f`, `tree`, `normal`, `monte_carlo_gbm`

**Metric naming:**

- Simple: `dv01`, `duration`, `convexity`, `ytm`
- Bucketed: `bucketed_dv01::USD-OIS::10y`
- Credit-style: `cs01::BOND_A`

In [17]:
model_keys = (
    "discounting",
    "black76",
    "hazard_rate",
    "hull_white_1f",
    "tree",
    "normal",
    "monte_carlo_gbm",
)
print("Supported model keys (from curriculum):")
for k in model_keys:
    print(f"  {k}")
print()
examples = [
    "dv01",
    "duration",
    "bucketed_dv01::USD-OIS::10y",
    "cs01::BOND_A",
]
print("Example metric id shapes:")
for e in examples:
    print(f"  {e}")

Supported model keys (from curriculum):
  discounting
  black76
  hazard_rate
  hull_white_1f
  tree
  normal
  monte_carlo_gbm

Example metric id shapes:
  dv01
  duration
  bucketed_dv01::USD-OIS::10y
  cs01::BOND_A


## Mini-example: price a deposit (full pipeline)

1. Define instrument JSON  
2. Validate with `validate_instrument_json`  
3. Build `MarketContext`, serialize with `to_json()`  
4. Call `price_instrument_with_metrics` with **`discounting`** and deposit-relevant metrics  
5. Parse `ValuationResult` and print PV plus each metric

In [18]:
from datetime import date

import json

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.valuations import (
    ValuationResult,
    price_instrument,
    price_instrument_with_metrics,
    validate_instrument_json,
)

as_of = "2025-01-15"

# 1) Instrument JSON
raw = json.dumps(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-MINI",
            "notional": {"amount": 1_000_000.0, "currency": "USD"},
            "start_date": as_of,
            "maturity": "2025-06-15",
            "day_count": "Act360",
            "quote_rate": 0.05,
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)

# 2) Validate
instrument_json = validate_instrument_json(raw)
print("Step 2 — canonical instrument (first 220 chars):")
print(instrument_json[:220] + "...")

# 3) Market snapshot
mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date.fromisoformat(as_of),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75)],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
print("Step 3 — market_json ready:", len(market_json), "chars")

# 4) Price (PV only, then with metrics)
pv_only = price_instrument(instrument_json, market_json, as_of, model="discounting")
print("Step 4a — raw PV JSON (one line):", pv_only.splitlines()[0][:120], "...")

metrics = ["dv01", "deposit_par_rate", "quote_rate", "yf"]
priced = price_instrument_with_metrics(
    instrument_json,
    market_json,
    as_of,
    model="discounting",
    metrics=metrics,
)

# 5) Parse envelope
vr = ValuationResult.from_json(priced)
print("Step 5 — ValuationResult:")
print(f"  id={vr.instrument_id!r}  pv={vr.price:.2f} {vr.currency}")
print(f"  metrics requested: {metrics}")
print(f"  metrics returned: {vr.metric_keys()}")
for key in sorted(vr.metric_keys()):
    print(f"    {key}: {vr.get_metric(key)}")

Step 2 — canonical instrument (first 220 chars):
{
  "type": "deposit",
  "spec": {
    "id": "DEP-MINI",
    "notional": {
      "amount": "1000000",
      "currency": "USD"
    },
    "start_date": "2025-01-15",
    "maturity": "2025-06-15",
    "day_count": "Act360"...
Step 3 — market_json ready: 756 chars
Step 4a — raw PV JSON (one line): { ...
Step 5 — ValuationResult:
  id='DEP-MINI'  pv=-145.58 USD
  metrics requested: ['dv01', 'deposit_par_rate', 'quote_rate', 'yf']
  metrics returned: ['dv01', 'deposit_par_rate', 'quote_rate', 'yf']
    deposit_par_rate: 0.050354409049918725
    dv01: -41.363840395655274
    quote_rate: 0.05
    yf: 0.41944444444444445


### Report components for downstream presentation

The pricing pipeline does not stop at PV and Greeks. The helpers below turn analytics into structured payloads that can feed notebooks, markdown reports, or JSON APIs without custom post-processing.


In [ ]:
from datetime import date, timedelta

from finstack.valuations import (
    cashflow_ladder,
    format_bps,
    format_currency,
    format_pct,
    format_ratio,
    metrics_table_from_dict,
    scenario_matrix,
    waterfall_from_steps,
)

metrics_component = metrics_table_from_dict(
    instrument_id="ACME-5Y-SNR",
    as_of="2026-04-16",
    currency="USD",
    npv=1_002_350.0,
    metrics={
        "dv01": 425.37,
        "cs01": 180.12,
        "ytm": 0.0475,
        "modified_duration": 4.25,
        "convexity": 22.8,
    },
)

first = date(2026, 6, 15)
dates = [(first + timedelta(days=91 * idx)).isoformat() for idx in range(20)]
ladder_component = cashflow_ladder(
    instrument_id="ACME-5Y-SNR",
    currency="USD",
    dates=dates,
    principal=[0.0] * 19 + [1_000_000.0],
    interest=[11_875.0] * 20,
    frequency="quarterly",
)

scenario_component = scenario_matrix(
    title="ACME-5Y-SNR scenario grid",
    scenarios=[
        ("Base", {"npv": 1_000_000.0, "dv01": 425.0, "ytm": 0.0475, "oas": 0.0125}),
        ("Upside", {"npv": 1_035_000.0, "dv01": 430.0, "ytm": 0.0425, "oas": 0.0105}),
        ("Downside", {"npv": 945_000.0, "dv01": 418.0, "ytm": 0.0565, "oas": 0.0185}),
    ],
    base_case="Base",
)

waterfall_component = waterfall_from_steps(
    title="ACME-5Y-SNR daily P&L",
    currency="USD",
    start_value=1_000_000.0,
    end_value=1_029_400.0,
    steps=[("Rates", 18_250.0), ("Credit", 12_400.0), ("Vol", -3_100.0), ("Basis", 1_850.0)],
)

print(metrics_component["markdown"])
print(ladder_component["markdown"].splitlines()[0])
print(scenario_component["markdown"].splitlines()[0])
print(waterfall_component["markdown"].splitlines()[0])
print("\nFormatting helpers")
print("  ", format_bps(0.0025, 1), format_pct(0.0534, 2), format_currency(1_234_567.89, "USD", 2), format_ratio(3.5, 2))
print("\nJSON payload keys:")
print("  metrics  ->", sorted(metrics_component["json"].keys()))
print("  ladder   ->", sorted(ladder_component["json"].keys()))
print("  scenario ->", sorted(scenario_component["json"].keys()))
print("  waterfall->", sorted(waterfall_component["json"].keys()))


### Valuation cache mechanics

`ValuationCache` is a lightweight way to memoize repeated lookups keyed by instrument and market version. The example below shows the hit-rate improvement on repeated pricing, the miss reset after a market-version bump, and an eviction / targeted invalidation sanity check.


In [ ]:
from finstack.valuations import ValuationCache


def fake_npv(instrument_id: int, market_version: int) -> float:
    return 1_000_000.0 + instrument_id * 137.0 - market_version * 53.0


cache = ValuationCache(max_entries=1_000, max_memory_bytes=256_000_000)
for _pass in range(10):
    for inst_id in range(100):
        hit = cache.get(inst_id, 1)
        if hit is None:
            cache.insert(inst_id, fake_npv(inst_id, 1), 1)

stats_after_repricing = cache.stats()
print("After repeated repricing:")
print(stats_after_repricing)

for inst_id in range(100):
    assert cache.get(inst_id, 2) is None

stats_after_version_bump = cache.stats()
print("\nAfter market-version bump:")
print(stats_after_version_bump)

small_cache = ValuationCache(max_entries=50, max_memory_bytes=256_000_000)
for inst_id in range(100):
    small_cache.insert(inst_id, fake_npv(inst_id, 1), 1)

invalidate_cache = ValuationCache(max_entries=1_000, max_memory_bytes=256_000_000)
for mv in range(1, 4):
    invalidate_cache.insert(42, fake_npv(42, mv), mv)
for inst_id in (1, 2, 3):
    invalidate_cache.insert(inst_id, fake_npv(inst_id, 1), 1)
invalidate_cache.invalidate_instrument(42)

print("\nEviction demo:")
print(small_cache.stats())
print("\nTargeted invalidation demo:")
print(invalidate_cache.stats())


## Takeaways

- **Tagged instrument JSON** plus **`validate_instrument_json`** gives you a stable, loader-ready payload.
- **`MarketContext.to_json()`** produces the market snapshot string expected by **`price_instrument`** / **`price_instrument_with_metrics`**.
- Choose a **model key** (`discounting` is the workhorse for curve-discounted cashflows).
- **`ValuationResult.from_json`** is the ergonomic read API for PV, currency, covenant flags, and **`measures`**.
- **Metrics are instrument-specific** — use `list_standard_metrics_grouped` to browse by category; irrelevant IDs are omitted rather than erroring.

**Next:** continue the pricing track with instruments that need forward curves, vol, or credit (swaps, options, CDS) using the same JSON → market → model pattern.